# Parse Vitis HLS `csynth.xml` with PySilicon

This notebook mirrors the course `Untitled.zip` flow for reading the Vitis HLS C-synthesis XML report.

It extracts:

- loop pipeline II / depth / latency information
- resource utilization

Expected generated report locations:

```text
jpegls_hls_prj/solution1/syn/report/csynth.xml
jpegls_hls_prj/solution1/syn/report/jpegls_encode_hls_csynth.xml
```

For a Vitis Unified IDE style component, the analogous path may be:

```text
hls_component/solution1/syn/reports/csynth.xml
```


In [ ]:
from pathlib import Path
import os
import pandas as pd

repo_root = Path.cwd()
data_dir = repo_root / "data"
data_dir.mkdir(exist_ok=True)

candidate_solution_paths = [
    repo_root / "jpegls_hls_prj" / "solution1",
    repo_root / "jpegls_hls_cosim_prj" / "solution1",
    repo_root / "hls_component" / "solution1",
]

sol_path = None
for candidate in candidate_solution_paths:
    if candidate.exists():
        sol_path = candidate
        break

print("Repository root:", repo_root)
print("Selected solution path:", sol_path)


## PySilicon parser

This is the same style as the line-intersection notebook:

```python
from pysilicon.utils.csynthparse import CsynthParser
parser = CsynthParser(sol_path=sol_path)
parser.get_loop_pipeline_info()
parser.get_resources()
```


In [ ]:
from pysilicon.utils.csynthparse import CsynthParser

if sol_path is None:
    raise FileNotFoundError(
        "No generated HLS solution directory was found. "
        "Run `vitis_hls -f hls/run_hls.tcl` first."
    )

parser = CsynthParser(sol_path=str(sol_path))

print("Latency and Initiation Interval:")
parser.get_loop_pipeline_info()
with pd.option_context("display.max_columns", None, "display.width", 300):
    display(parser.loop_df)

print("\nResource Usage:")
parser.get_resources()
display(parser.res_df)


In [ ]:
# Write the parser outputs to CSV files in the data directory.
# These filenames match the course notebook style.

parser.loop_df.to_csv(data_dir / "csynth_loop_info.csv", index=True)
parser.res_df.to_csv(data_dir / "csynth_resource_usage.csv", index=False)

print("Wrote:", data_dir / "csynth_loop_info.csv")
print("Wrote:", data_dir / "csynth_resource_usage.csv")


## Command-line equivalent

The repository also includes:

```bash
python scripts/parse_csynth_pysilicon.py
```

That script tries PySilicon first and then falls back to a local XML parser if PySilicon is not installed.
